# Lab 01-02: Model Task Selection

**Module:** 01 -- Design Applications (14% of exam, ~6 questions)  
**UI Sections:** Playground, Discover  
**Estimated Time:** 30--40 minutes  
**Difficulty:** Beginner

## What You Will Learn

Different NLP tasks need different models. You wouldn't use a hammer to screw in a bolt. Same with LLMs -- a summarization task and a classification task might use different approaches, different prompt patterns, and sometimes entirely different model architectures.

The exam tests whether you can **map business requirements to the correct task type**. This sounds simple, but the questions are designed to confuse you with overlapping categories and one particularly sneaky distractor ("Sentencizer" -- more on that later).

In this lab, you will:

1. Learn the seven core NLP task types and when to use each
2. Practice mapping real business scenarios to the correct task type
3. Call an LLM with prompts designed for each task type
4. Understand the "Sentencizer" trap that appears on the exam

## Mental Model

> **Analogy -- NLP tasks as kitchen appliances:**
>
> Think of NLP tasks like kitchen appliances. Each one does a specific job, and you pick the appliance based on what you are making:
>
> | Appliance | NLP Task | What It Does |
> |---|---|---|
> | **Blender** | Summarization | Takes a lot of ingredients and reduces them to a smooth result |
> | **Label maker** | Text Classification | Puts things into categories |
> | **Writing desk** | Text Generation | Produces something new from a prompt/idea |
> | **Reference librarian** | Question Answering | Finds the answer within a body of knowledge |
> | **X-ray machine** | Feature Extraction | Reveals the hidden internal structure (embeddings) |
> | **Highlighter** | Token Classification | Marks specific parts of text (names, dates, places) |
> | **Translator** | Text2Text Generation | Transforms from one form to another |
>
> You would not put bread in a blender to make toast. Similarly, you would not use a summarization model to extract named entities. **Picking the right appliance is half the job.**

## Exam Alert

| Trap | Correct Answer |
|---|---|
| "Sentencizer" is a valid NLP/LLM task type | **WRONG** -- Sentencizer is a spaCy pipeline component for sentence boundary detection, NOT an LLM task |
| Feature extraction means pulling keywords from text | **WRONG** -- In the LLM context, feature extraction means generating **embeddings** (dense vector representations) |
| Token classification is about counting tokens | **WRONG** -- Token classification labels individual tokens (words) with categories like person, location, organization |
| Text2Text and text generation are the same thing | **WRONG** -- Text2Text transforms input text to output text (translation, paraphrasing). Text generation creates new content from a prompt |
| Summarization and text generation are interchangeable | **WRONG** -- Summarization specifically condenses; text generation broadly creates new text |

> **Exam tip:** When you see "Sentencizer" as an answer choice, eliminate it immediately. It is the most common distractor on task-selection questions.

## Prerequisites & UI Navigation

- Completed Lab 00-03 (first LLM call -- you know how to use the OpenAI-compatible SDK)
- Access to Foundation Model APIs

### Explore the Playground and Discover

1. **UI -->** Left nav --> **Playground** (chat bubble icon)
2. In the model dropdown, select **databricks-meta-llama-3-3-70b-instruct**
3. Try asking: `Summarize the following: The quick brown fox jumps over the lazy dog. The dog did not seem to mind.`
4. Now try: `Classify the sentiment of this review: The food was amazing but the service was slow.`
5. Notice how the **same model** handles different task types based on the prompt

To explore available models:

1. **UI -->** Left nav --> **Discover** (or **Machine Learning --> Models**)
2. Browse the model catalog -- each model card shows what task types it supports
3. Filter by task type (e.g., "text-generation", "feature-extraction") to see specialized models

In [ ]:
# ------------------------------------------------------------------
# Setup: Install dependencies
# ------------------------------------------------------------------

%pip install openai --quiet

# Restart Python to pick up the new package
dbutils.library.restartPython()

**Expected output:** Installation logs followed by `Python interpreter will be restarted.`

This is normal -- proceed to the next cell.

In [ ]:
# ------------------------------------------------------------------
# Setup: Initialize the OpenAI-compatible client for Databricks
# ------------------------------------------------------------------

from openai import OpenAI
import os

client = OpenAI(
    api_key=os.environ.get("DATABRICKS_TOKEN"),
    base_url=f"{os.environ.get('DATABRICKS_HOST')}/serving-endpoints"
)

MODEL = "databricks-meta-llama-3-3-70b-instruct"

print("Client initialized successfully.")
print(f"Model: {MODEL}")
print(f"Endpoint: {os.environ.get('DATABRICKS_HOST')}/serving-endpoints")

**Expected output:**
```
Client initialized successfully.
Model: databricks-meta-llama-3-3-70b-instruct
Endpoint: https://<your-workspace>.cloud.databricks.com/serving-endpoints
```

---

## Step 1: The NLP Task Taxonomy

There are seven core NLP task types you need to know for the exam. Each one has a distinct purpose, common use cases, and a characteristic prompt pattern. Let's build a comprehensive reference table.

Think of this table as your "menu" -- when the exam gives you a business scenario, you look at this menu and pick the dish that fits.

In [ ]:
# ------------------------------------------------------------------
# Step 1: Build the NLP task taxonomy as a reference DataFrame
# ------------------------------------------------------------------

import pandas as pd

task_taxonomy = pd.DataFrame([
    {
        "Task Type": "Summarization",
        "Description": "Condense long text into a shorter version that preserves key information",
        "Example Use Cases": "Meeting notes digest, article summaries, contract highlights",
        "Example Prompt Pattern": "Summarize the following text in 3 sentences: {text}",
        "Kitchen Analogy": "Blender -- reduces many ingredients into a smooth result"
    },
    {
        "Task Type": "Text Classification",
        "Description": "Assign one or more labels/categories to a piece of text",
        "Example Use Cases": "Sentiment analysis, spam detection, support ticket routing",
        "Example Prompt Pattern": "Classify the sentiment of this review as positive, negative, or neutral: {text}",
        "Kitchen Analogy": "Label maker -- sorts things into categories"
    },
    {
        "Task Type": "Text Generation",
        "Description": "Produce new text from a prompt -- the most general task type",
        "Example Use Cases": "Drafting emails, writing code, creative content, chatbot responses",
        "Example Prompt Pattern": "Write a professional email declining a meeting invitation.",
        "Kitchen Analogy": "Writing desk -- creates something new from an idea"
    },
    {
        "Task Type": "Question Answering",
        "Description": "Answer a question based on provided context or general knowledge",
        "Example Use Cases": "FAQ bots, document Q&A, RAG pipelines, customer support",
        "Example Prompt Pattern": "Based on the following context, answer the question.\nContext: {context}\nQuestion: {question}",
        "Kitchen Analogy": "Reference librarian -- finds the answer in a body of knowledge"
    },
    {
        "Task Type": "Feature Extraction",
        "Description": "Generate embeddings (dense vector representations) from text",
        "Example Use Cases": "Semantic search, similarity matching, clustering, vector databases",
        "Example Prompt Pattern": "(No prompt -- you call an embedding endpoint that returns a vector)",
        "Kitchen Analogy": "X-ray machine -- reveals hidden internal structure"
    },
    {
        "Task Type": "Token Classification",
        "Description": "Label individual tokens (words/subwords) within text",
        "Example Use Cases": "Named Entity Recognition (NER), POS tagging, detecting PII",
        "Example Prompt Pattern": "Identify all person names, organizations, and locations in this text: {text}",
        "Kitchen Analogy": "Highlighter -- marks specific parts of text"
    },
    {
        "Task Type": "Text2Text Generation",
        "Description": "Transform text from one form to another (input text --> output text)",
        "Example Use Cases": "Translation, paraphrasing, grammar correction, format conversion",
        "Example Prompt Pattern": "Translate the following English text to French: {text}",
        "Kitchen Analogy": "Translator -- transforms from one form to another"
    }
])

# Display the full table nicely
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_columns", 6)
display(task_taxonomy)

**Expected output:** A formatted table with 7 rows -- one per task type -- showing the description, use cases, prompt pattern, and kitchen analogy for each.

### Key Distinctions to Memorize

Some task types overlap, and the exam exploits this. Here are the critical boundaries:

| Confusion Pair | How to Tell Them Apart |
|---|---|
| **Summarization** vs **Text Generation** | Summarization takes long input and produces shorter output. Text generation creates new content that may not have existed in the input. |
| **Text Classification** vs **Token Classification** | Text classification labels the *entire* text ("this email is spam"). Token classification labels *individual words* ("John = PERSON, Paris = LOCATION"). |
| **Text Generation** vs **Text2Text Generation** | Text generation creates from a prompt. Text2Text transforms existing text into a new form (translation, paraphrasing). Both produce text, but Text2Text always has meaningful input text being transformed. |
| **Question Answering** vs **Text Generation** | QA extracts/finds an answer from context. Text generation can answer questions too, but QA specifically implies grounding in provided context. |
| **Feature Extraction** vs everything else | Feature extraction does NOT produce human-readable text. It produces numerical vectors (embeddings). If the output is numbers, it is feature extraction. |

---

## Step 2: Business Requirement --> Task Mapping Practice

This is the core exam skill. You will be given a business scenario and asked: *"Which NLP task type best fits this requirement?"*

Let's practice with six realistic scenarios that mirror the style of exam questions. First, we will present them as a quiz, then explain each answer.

In [ ]:
# ------------------------------------------------------------------
# Step 2: Interactive quiz -- map business scenarios to task types
# ------------------------------------------------------------------

# Define the six scenarios the exam commonly tests
scenarios = [
    {
        "id": 1,
        "scenario": (
            "A legal team has 500-page contracts and needs a 1-page executive "
            "summary of each contract highlighting key terms and obligations."
        ),
        "answer": "Summarization",
        "explanation": (
            "The task is to CONDENSE long text into shorter text while preserving "
            "key information. This is the textbook definition of summarization. "
            "The input is long (500 pages), the output is short (1 page)."
        )
    },
    {
        "id": 2,
        "scenario": (
            "A customer support team receives thousands of tickets daily and needs "
            "each ticket automatically routed to the right department (billing, "
            "technical, shipping, general)."
        ),
        "answer": "Text Classification",
        "explanation": (
            "The task is to assign a CATEGORY (billing, technical, shipping, general) "
            "to each ticket. This is text classification -- the model reads the text "
            "and outputs a label. The key signal is 'routing to departments' = categories."
        )
    },
    {
        "id": 3,
        "scenario": (
            "A healthcare company needs to scan clinical notes and identify all "
            "mentions of patient names, medication names, and dosages."
        ),
        "answer": "Token Classification",
        "explanation": (
            "The task is to label INDIVIDUAL WORDS (tokens) within the text -- "
            "'Dr. Smith' = PERSON, 'Aspirin' = MEDICATION, '500mg' = DOSAGE. "
            "This is Named Entity Recognition (NER), which is a type of token "
            "classification. The key signal is 'identify mentions' of specific entity types."
        )
    },
    {
        "id": 4,
        "scenario": (
            "An e-commerce company wants to build a semantic search engine where "
            "users can search products by meaning (e.g., 'comfortable shoes for "
            "standing all day') rather than exact keyword match."
        ),
        "answer": "Feature Extraction",
        "explanation": (
            "Semantic search requires EMBEDDINGS -- vector representations of text that "
            "capture meaning. The model converts product descriptions and search queries "
            "into vectors, then finds the closest matches. Feature extraction = embeddings. "
            "The key signal is 'semantic search' or 'meaning-based similarity'."
        )
    },
    {
        "id": 5,
        "scenario": (
            "A multinational company needs to convert all their English product "
            "manuals into Spanish, French, and German for regional markets."
        ),
        "answer": "Text2Text Generation",
        "explanation": (
            "Translation is the canonical example of Text2Text generation -- you have "
            "input text in one form (English) and you transform it into another form "
            "(Spanish/French/German). The meaning is preserved, but the form changes. "
            "The key signal is 'convert' or 'translate' from one form to another."
        )
    },
    {
        "id": 6,
        "scenario": (
            "A knowledge base has 10,000 articles. Users ask natural language "
            "questions and the system should find the answer within the relevant "
            "article and return a direct answer."
        ),
        "answer": "Question Answering",
        "explanation": (
            "The task is to ANSWER A QUESTION based on a provided CONTEXT (the articles). "
            "This is extractive or generative QA -- the model reads the article and "
            "produces an answer grounded in that context. The key signal is 'ask questions' "
            "+ 'find the answer within' a document."
        )
    }
]


def run_task_quiz(scenarios):
    """Run an interactive quiz that presents business scenarios and checks answers."""
    valid_tasks = [
        "Summarization", "Text Classification", "Text Generation",
        "Question Answering", "Feature Extraction",
        "Token Classification", "Text2Text Generation"
    ]
    score = 0
    total = len(scenarios)

    print("=" * 70)
    print("  TASK MAPPING QUIZ -- Map each scenario to the correct NLP task")
    print("=" * 70)
    print(f"\nValid task types: {', '.join(valid_tasks)}\n")

    for s in scenarios:
        print(f"--- Scenario {s['id']} of {total} ---")
        print(f"  {s['scenario']}")
        print()

        # Auto-reveal answer (in a live lab, you would use input() here)
        print(f"  CORRECT ANSWER: {s['answer']}")
        print(f"  WHY: {s['explanation']}")
        print()
        score += 1

    print("=" * 70)
    print(f"  Review complete: {total} scenarios covered")
    print("=" * 70)


run_task_quiz(scenarios)

**Expected output:** Six business scenarios, each with the correct task type and an explanation of why that task applies.

```
======================================================================
  TASK MAPPING QUIZ -- Map each scenario to the correct NLP task
======================================================================

Valid task types: Summarization, Text Classification, Text Generation, ...

--- Scenario 1 of 6 ---
  A legal team has 500-page contracts and needs a 1-page executive ...

  CORRECT ANSWER: Summarization
  WHY: The task is to CONDENSE long text into shorter text ...

--- Scenario 2 of 6 ---
  ...
```

### Exam Signal Words

When reading exam questions, look for these signal words to identify the task type:

| Signal Words | Task Type |
|---|---|
| "condense", "shorten", "summarize", "digest", "brief" | Summarization |
| "categorize", "classify", "label", "route", "sentiment" | Text Classification |
| "write", "generate", "draft", "create", "compose" | Text Generation |
| "answer", "find in context", "extract answer from" | Question Answering |
| "embedding", "vector", "semantic search", "similarity" | Feature Extraction |
| "identify entities", "NER", "tag words", "label tokens" | Token Classification |
| "translate", "paraphrase", "convert", "rewrite" | Text2Text Generation |

---

## Step 3: Demonstrate Each Task Type with LLM Calls

A powerful insight: **a single general-purpose LLM can handle most of these task types.** The difference is in the prompt, not the model. The model `databricks-meta-llama-3-3-70b-instruct` is a text generation model, but with the right prompt it can summarize, classify, answer questions, and more.

The exception is **Feature Extraction** -- that requires an embedding model (like `databricks-gte-large-en`), not a chat model.

Let's call the LLM once for each task type (except feature extraction, which we will handle separately) and see how prompt design drives the behavior.

In [ ]:
# ------------------------------------------------------------------
# Step 3a: Demonstrate task types using the same LLM with different prompts
# ------------------------------------------------------------------

# Sample text we will use across multiple task demonstrations
sample_text = (
    "Apple Inc. reported record quarterly revenue of $123.9 billion for Q1 2024, "
    "driven by strong iPhone sales in emerging markets. CEO Tim Cook announced "
    "plans to expand manufacturing operations in Vietnam and India. The company "
    "also unveiled a new AI initiative called 'Apple Intelligence' that will be "
    "integrated across all devices starting with iOS 18. Analysts from Goldman "
    "Sachs and Morgan Stanley maintained their 'buy' ratings, citing strong "
    "services revenue growth of 11% year-over-year. However, sales in China "
    "declined 13% due to increased competition from Huawei."
)

# Define prompts for each task type
task_demos = {
    "Summarization": (
        f"Summarize the following text in exactly 2 sentences:\n\n{sample_text}"
    ),
    "Text Classification": (
        f"Classify the overall sentiment of the following text as POSITIVE, "
        f"NEGATIVE, or MIXED. Respond with only the label.\n\n{sample_text}"
    ),
    "Text Generation": (
        "Write a 3-sentence LinkedIn post announcing a new AI product launch "
        "for a tech company. Keep it professional and exciting."
    ),
    "Question Answering": (
        f"Based on the following context, answer the question.\n\n"
        f"Context: {sample_text}\n\n"
        f"Question: Why did sales decline in China?"
    ),
    "Token Classification (NER)": (
        f"Identify all named entities in the following text. For each entity, "
        f"provide the entity text and its type (PERSON, ORGANIZATION, LOCATION, "
        f"DATE, MONEY). Format as a list.\n\n{sample_text}"
    ),
    "Text2Text Generation": (
        f"Translate the following text to French:\n\n"
        f"Apple reported record revenue driven by strong iPhone sales. "
        f"The company plans to expand operations in Vietnam and India."
    )
}

# Call the LLM for each task
print("=" * 70)
print("  SAME MODEL, DIFFERENT TASKS -- Prompt is what drives behavior")
print("=" * 70)

for task_name, prompt in task_demos.items():
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a precise assistant. Follow instructions exactly."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=200,
        temperature=0.0
    )
    result = response.choices[0].message.content.strip()
    print(f"\n--- {task_name} ---")
    print(f"Result: {result}")

**Expected output:**
```
======================================================================
  SAME MODEL, DIFFERENT TASKS -- Prompt is what drives behavior
======================================================================

--- Summarization ---
Result: Apple Inc. reported a record Q1 2024 revenue of $123.9 billion, 
fueled by strong iPhone sales and 11% services growth. However, the 
company faced a 13% decline in China due to Huawei competition.

--- Text Classification ---
Result: MIXED

--- Text Generation ---
Result: We are thrilled to announce the launch of our groundbreaking AI 
product that will transform how businesses operate. After years of 
research and development, we are bringing cutting-edge intelligence to 
your fingertips. Stay tuned for exciting updates as we roll out across 
all platforms!

--- Question Answering ---
Result: Sales in China declined 13% due to increased competition from Huawei.

--- Token Classification (NER) ---
Result:
- Apple Inc. (ORGANIZATION)
- Q1 2024 (DATE)
- $123.9 billion (MONEY)
- Tim Cook (PERSON)
- Vietnam (LOCATION)
- India (LOCATION)
- Goldman Sachs (ORGANIZATION)
- Morgan Stanley (ORGANIZATION)
- China (LOCATION)
- Huawei (ORGANIZATION)

--- Text2Text Generation ---
Result: Apple a annonce un chiffre d'affaires record, porte par de 
fortes ventes d'iPhone. L'entreprise prevoit d'etendre ses operations 
au Vietnam et en Inde.
```

(Your exact wording will vary, but the structure should match.)

**Key insight:** All six responses came from the **same model** (`databricks-meta-llama-3-3-70b-instruct`). The task type is determined by the prompt, not the model. However, for production systems, specialized models (like a fine-tuned NER model) often outperform general-purpose LLMs on specific tasks.

In [ ]:
# ------------------------------------------------------------------
# Step 3b: Feature Extraction (embeddings) -- requires an embedding model
# ------------------------------------------------------------------

# Feature extraction is different: instead of generating text,
# it generates a numerical vector (embedding) that captures the
# meaning of the text. This is used for semantic search, similarity,
# and vector databases.

EMBEDDING_MODEL = "databricks-gte-large-en"

texts_to_embed = [
    "Apple reported record revenue from iPhone sales.",
    "The tech company saw strong smartphone earnings.",
    "I love cooking Italian pasta on weekends."
]

embedding_response = client.embeddings.create(
    model=EMBEDDING_MODEL,
    input=texts_to_embed
)

print("=== Feature Extraction (Embeddings) ===")
print(f"Model: {EMBEDDING_MODEL}")
print(f"Number of texts embedded: {len(texts_to_embed)}")
print(f"Embedding dimension: {len(embedding_response.data[0].embedding)}")
print()

for i, text in enumerate(texts_to_embed):
    vec = embedding_response.data[i].embedding
    print(f"Text {i+1}: \"{text}\"")
    print(f"  Vector (first 5 dims): {[round(v, 4) for v in vec[:5]]}...")
    print()

# Compute cosine similarity to show semantic closeness
import math

def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(x * x for x in b))
    return dot / (norm_a * norm_b) if norm_a and norm_b else 0.0

vec_0 = embedding_response.data[0].embedding
vec_1 = embedding_response.data[1].embedding
vec_2 = embedding_response.data[2].embedding

sim_01 = cosine_similarity(vec_0, vec_1)
sim_02 = cosine_similarity(vec_0, vec_2)

print("=== Semantic Similarity ===")
print(f"Text 1 vs Text 2 (both about tech revenue): {sim_01:.4f}")
print(f"Text 1 vs Text 3 (tech revenue vs cooking):  {sim_02:.4f}")
print()
print("The two tech-related sentences are semantically closer (higher similarity)")
print("than the tech sentence vs the cooking sentence. This is what powers")
print("semantic search -- find documents with similar MEANING, not just keywords.")

**Expected output:**
```
=== Feature Extraction (Embeddings) ===
Model: databricks-gte-large-en
Number of texts embedded: 3
Embedding dimension: 1024

Text 1: "Apple reported record revenue from iPhone sales."
  Vector (first 5 dims): [0.0123, -0.0456, 0.0789, ...]...

Text 2: "The tech company saw strong smartphone earnings."
  Vector (first 5 dims): [0.0134, -0.0423, 0.0801, ...]...

Text 3: "I love cooking Italian pasta on weekends."
  Vector (first 5 dims): [-0.0234, 0.0567, -0.0123, ...]...

=== Semantic Similarity ===
Text 1 vs Text 2 (both about tech revenue): 0.8934
Text 1 vs Text 3 (tech revenue vs cooking):  0.4521

The two tech-related sentences are semantically closer (higher similarity)
than the tech sentence vs the cooking sentence.
```

(Your exact numbers will vary, but Text 1 vs Text 2 should have notably higher similarity than Text 1 vs Text 3.)

**What just happened:** Feature extraction does not produce human-readable text. It produces a dense numerical vector (1024 dimensions for GTE-Large). These vectors are the foundation of semantic search, RAG, and vector databases. When the exam mentions "embeddings" or "vector representations," the answer is feature extraction.

---

## Step 4: The "Sentencizer" Trap

This is the single most common distractor on task-selection questions. The exam will list answer choices like:

```
A) Summarization
B) Text Classification
C) Sentencizer
D) Token Classification
```

**"Sentencizer" is NEVER the correct answer.** It is not an NLP task type. It is a spaCy pipeline component that detects sentence boundaries.

### What Is a Sentencizer, Actually?

In spaCy (a popular NLP library), a **Sentencizer** is a rule-based component that splits text into sentences. It looks for punctuation marks (periods, exclamation points, question marks) and splits the text there. It is a **pre-processing utility**, not a task you would ask an LLM to perform.

| Aspect | Sentencizer (spaCy) | LLM Task Types |
|---|---|---|
| **What it is** | A pipeline component for sentence boundary detection | Categories of what models can do |
| **Library** | spaCy | HuggingFace, Databricks, OpenAI, etc. |
| **Output** | List of sentence spans | Depends on task (text, labels, vectors) |
| **Uses ML?** | No -- it is rule-based (looks for punctuation) | Yes -- uses transformer models |
| **Exam relevance** | It is a DISTRACTOR (wrong answer) | These are the CORRECT answer categories |

In [ ]:
# ------------------------------------------------------------------
# Step 4a: What Sentencizer actually does (spaCy sentence splitting)
# ------------------------------------------------------------------

# NOTE: This cell demonstrates what Sentencizer IS -- a simple rule-based
# sentence splitter, NOT an LLM task. We're showing this so you understand
# why it's a trap answer on the exam.

# You do NOT need spaCy installed to understand this concept.
# Here's what the Sentencizer does, implemented from scratch:

def simple_sentencizer(text):
    """Rule-based sentence boundary detection -- this is what spaCy's
    Sentencizer does. It splits text on punctuation boundaries.
    No ML involved -- just pattern matching."""
    import re
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s for s in sentences if s]

sample = (
    "Apple Inc. reported record revenue. Tim Cook announced expansion plans. "
    "The stock price rose 3% in after-hours trading! Will the trend continue?"
)

sentences = simple_sentencizer(sample)

print("=== Sentencizer Demo (Rule-Based Sentence Splitting) ===")
print(f"Input text: \"{sample}\"")
print(f"\nDetected {len(sentences)} sentences:")
for i, sent in enumerate(sentences, 1):
    print(f"  {i}. {sent}")

print("\n" + "=" * 70)
print("NOTICE: This is just text splitting. No understanding, no intelligence.")
print("A Sentencizer does NOT classify, summarize, generate, or answer anything.")
print("It is a pre-processing utility, NOT a model task.")
print("=" * 70)

**Expected output:**
```
=== Sentencizer Demo (Rule-Based Sentence Splitting) ===
Input text: "Apple Inc. reported record revenue. Tim Cook announced ..."

Detected 4 sentences:
  1. Apple Inc. reported record revenue.
  2. Tim Cook announced expansion plans.
  3. The stock price rose 3% in after-hours trading!
  4. Will the trend continue?

======================================================================
NOTICE: This is just text splitting. No understanding, no intelligence.
A Sentencizer does NOT classify, summarize, generate, or answer anything.
It is a pre-processing utility, NOT a model task.
======================================================================
```

Note how the simple rule-based approach has a limitation: it incorrectly splits on "Inc." because it sees a period. The real spaCy Sentencizer has better rules for abbreviations, but it is still fundamentally rule-based, not ML-based.

In [ ]:
# ------------------------------------------------------------------
# Step 4b: Contrast -- what an actual LLM task does with the same text
# ------------------------------------------------------------------

# Let's compare: Sentencizer (rule-based splitting) vs actual NLP tasks
# applied to the same text. This shows the qualitative difference.

sample_for_comparison = (
    "Apple Inc. reported record revenue. Tim Cook announced expansion plans. "
    "The stock price rose 3% in after-hours trading."
)

comparison_tasks = {
    "Summarization": f"Summarize in one sentence: {sample_for_comparison}",
    "Text Classification": (
        f"Classify the sentiment as POSITIVE, NEGATIVE, or NEUTRAL. "
        f"Respond with only the label.\n\n{sample_for_comparison}"
    ),
    "Token Classification": (
        f"List all named entities (PERSON, ORG, PERCENTAGE) in this text:\n\n"
        f"{sample_for_comparison}"
    )
}

print("=== Sentencizer vs Actual NLP Tasks ===")
print(f"\nInput: \"{sample_for_comparison}\"\n")

# Sentencizer output
print("--- Sentencizer (rule-based, NOT an LLM task) ---")
sents = simple_sentencizer(sample_for_comparison)
for s in sents:
    print(f"  Sentence: {s}")
print("  (Just splits text. No understanding.)\n")

# Actual LLM task outputs
for task_name, prompt in comparison_tasks.items():
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=150,
        temperature=0.0
    )
    result = response.choices[0].message.content.strip()
    print(f"--- {task_name} (real LLM task) ---")
    print(f"  {result}")
    print()

**Expected output:**
```
=== Sentencizer vs Actual NLP Tasks ===

Input: "Apple Inc. reported record revenue. Tim Cook announced ..."

--- Sentencizer (rule-based, NOT an LLM task) ---
  Sentence: Apple Inc.
  Sentence: reported record revenue.
  Sentence: Tim Cook announced expansion plans.
  Sentence: The stock price rose 3% in after-hours trading.
  (Just splits text. No understanding.)

--- Summarization (real LLM task) ---
  Apple reported record revenue, with Tim Cook announcing expansion and 
  the stock rising 3% after hours.

--- Text Classification (real LLM task) ---
  POSITIVE

--- Token Classification (real LLM task) ---
  - Apple Inc. (ORG)
  - Tim Cook (PERSON)
  - 3% (PERCENTAGE)
```

**The contrast is clear:** Sentencizer does mechanical text splitting. LLM tasks involve understanding, reasoning, and producing intelligent output. That is why Sentencizer is not in the same category -- it is a utility, not a task.

---

## Exam Tips

| # | Tip | Details |
|---|---|---|
| 1 | **Eliminate Sentencizer immediately** | If you see "Sentencizer" as an answer choice, cross it out. It is NEVER a correct NLP task type. It is a spaCy component for sentence boundary detection. |
| 2 | **Feature extraction = embeddings** | Do not confuse this with "extracting features/keywords from text." In the GenAI context, feature extraction specifically means generating dense vector representations (embeddings). |
| 3 | **Read the scenario for the OUTPUT type** | The output tells you the task: a label = classification, shorter text = summarization, a vector = feature extraction, tagged words = token classification. |
| 4 | **Text2Text needs both input AND output text** | If the scenario describes transforming text from one form to another (translation, paraphrasing, grammar correction), it is Text2Text. If it is creating from scratch, it is text generation. |
| 5 | **One model, many tasks** | General-purpose chat models can handle most task types via prompt engineering. But embedding models (feature extraction) are a separate model type entirely. |

---

## Key Takeaways

### The Seven NLP Task Types
- **Summarization** -- condense long text into shorter text (blender)
- **Text Classification** -- assign labels/categories to text (label maker)
- **Text Generation** -- produce new text from a prompt (writing desk)
- **Question Answering** -- answer questions from provided context (reference librarian)
- **Feature Extraction** -- generate embeddings/vectors from text (X-ray machine)
- **Token Classification** -- label individual tokens like NER (highlighter)
- **Text2Text Generation** -- transform text from one form to another (translator)

### Critical Distinctions
- **Sentencizer is NOT a task type** -- it is a spaCy pipeline component (exam trap)
- **Feature extraction outputs numbers (vectors), not text** -- the only task type that does this
- **Text classification labels the whole text; token classification labels individual words**
- **Text2Text transforms existing text; text generation creates new content**

### Practical Skills
- The same general-purpose model handles multiple task types via different prompts
- Embedding models (like `databricks-gte-large-en`) are separate from chat models
- Signal words in exam questions point you to the correct task type
- Always look at the **output format** to confirm the task type

---

## Next Lab

**Lab 01-03: Chain Design** -- you will learn how to connect multiple LLM calls together into chains, where the output of one step feeds into the next. This is the foundation of building real GenAI applications that go beyond single prompts.